# Brent Volatility Forecasting with GARCH and LSTM

## Project description

This notebook is part of the Machine Learning project on Brent Crude Oil volatility forecasting.

The objective is to compare three approaches for forecasting the annualized 10-day future realized volatility of Brent Crude Oil:

1. a naive persistence baseline;
2. a GARCH(1,1) econometric model;
3. a Long Short-Term Memory neural network.

The underlying asset is Brent Crude Oil, retrieved from Yahoo Finance using the ticker `BZ=F`.

The target variable is the 10-day future realized volatility, computed from daily log returns and annualized using 252 trading days.

Particular attention is given to:

- respecting the chronological order of the data;
- avoiding information leakage;
- comparing all models on the same test period;
- ensuring that the notebook is reproducible.

In [179]:
# Core libraries
import os
import random
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Data source
import yfinance as yf

# Machine learning tools
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# GARCH model
from arch import arch_model

# Deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

print("Imports OK")
print("TensorFlow version:", tf.__version__)

Imports OK
TensorFlow version: 2.21.0


In [180]:
# Reproducibility

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seeds fixed.")

Random seeds fixed.


In [181]:
# Project parameters

TICKER = "BZ=F"

START_DATE = "2005-01-01"
END_DATE = "2024-12-31"

PRICE_COL = "Price"

FORECAST_HORIZON = 10
LSTM_SEQUENCE_LENGTH = 30
ANNUALIZATION_FACTOR = 252

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

print("Project parameters defined.")

Project parameters defined.


In [182]:
# Configuration summary

config = {
    "ticker": TICKER,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "price_column": PRICE_COL,
    "forecast_horizon": FORECAST_HORIZON,
    "lstm_sequence_length": LSTM_SEQUENCE_LENGTH,
    "annualization_factor": ANNUALIZATION_FACTOR,
    "train_size": TRAIN_SIZE,
    "validation_size": VAL_SIZE,
    "test_size": TEST_SIZE,
}

config

{'ticker': 'BZ=F',
 'start_date': '2005-01-01',
 'end_date': '2024-12-31',
 'price_column': 'Price',
 'forecast_horizon': 10,
 'lstm_sequence_length': 30,
 'annualization_factor': 252,
 'train_size': 0.7,
 'validation_size': 0.15,
 'test_size': 0.15}

In [183]:
# Project paths

PROJECT_ROOT = os.path.abspath("..")

DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

for directory in [DATA_RAW_DIR, DATA_PROCESSED_DIR, FIGURES_DIR, RESULTS_DIR]:
    os.makedirs(directory, exist_ok=True)

print("Project directories ready.")
print("Project root:", PROJECT_ROOT)

Project directories ready.
Project root: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch


## 2. Data loading and cleaning

In this section, we load the historical Brent Crude Oil futures data, inspect the available columns, select the relevant price column, clean the dataset and ensure that observations are sorted in chronological order.

The dataset contains daily observations of Brent futures prices from January 1, 2005 to December 31, 2024. The data were collected from Investing.com.

The file contains the following columns: `Date`, `Price`, `Open`, `High`, `Low`, `Vol.` and `Change %`.

Since the dataset does not contain a column named `Close` or `Adj Close`, the column `Price` is used as the daily closing price reference. This column corresponds to the closing price, or last available daily price, of Brent futures.

Before computing returns and volatility measures, the dataset is cleaned as follows:

- dates are converted to a proper datetime format;
- observations are sorted in chronological order;
- duplicated dates are removed if necessary;
- missing values in the price column are checked;
- the final sample period and number of observations are reported.

In [184]:
# Load raw data

raw_file_path = os.path.join(DATA_RAW_DIR, "d:\\Documents\\ESLSCA - MBA2 Finance de marché\\Machine Learning\\Projet LSTM\\repository\\brent-volatility-lstm-garch\\data\\raw\\brent_oil_futures_2005_2024.csv")

brent_raw = pd.read_csv(raw_file_path)

print("Raw dataset shape:", brent_raw.shape)
brent_raw.head()

Raw dataset shape: (5158, 7)


,Date,Price,Open,High,Low,Vol.,Change %
0,2005-01-04,41.04,39.40,41.25,38.81,40.10K,1.43%
1,2005-01-05,40.51,40.80,41.00,39.90,42.23K,-1.29%
2,2005-01-06,42.85,40.43,43.20,39.82,51.63K,5.78%
3,2005-01-07,43.18,42.75,43.75,42.20,29.64K,0.77%
4,2005-01-10,42.92,43.20,44.85,42.90,27.65K,-0.60%


In [185]:
# Inspect available columns

print("Available columns:")
print(brent_raw.columns.tolist())

brent_raw.info()

Available columns:
['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']
<class 'pandas.DataFrame'>
RangeIndex: 5158 entries, 0 to 5157
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      5158 non-null   str    
 1   Price     5158 non-null   float64
 2   Open      5158 non-null   float64
 3   High      5158 non-null   float64
 4   Low       5158 non-null   float64
 5   Vol.      5156 non-null   str    
 6   Change %  5158 non-null   str    
dtypes: float64(4), str(3)
memory usage: 282.2 KB


In [186]:
# Parse dates and sort chronologically

brent = brent_raw.copy()

brent["Date"] = pd.to_datetime(brent["Date"], errors="coerce")

brent = brent.sort_values("Date").reset_index(drop=True)

brent.head()

,Date,Price,Open,High,Low,Vol.,Change %
0,2005-01-04,41.04,39.40,41.25,38.81,40.10K,1.43%
1,2005-01-05,40.51,40.80,41.00,39.90,42.23K,-1.29%
2,2005-01-06,42.85,40.43,43.20,39.82,51.63K,5.78%
3,2005-01-07,43.18,42.75,43.75,42.20,29.64K,0.77%
4,2005-01-10,42.92,43.20,44.85,42.90,27.65K,-0.60%


In [187]:
brent.tail()

,Date,Price,Open,High,Low,Vol.,Change %
5153,2024-12-24,73.58,72.96,73.81,72.79,93.49K,1.31%
5154,2024-12-26,73.26,73.77,74.17,72.99,54.05K,-0.43%
5155,2024-12-27,74.17,73.16,74.29,73.05,82.60K,1.24%
5156,2024-12-30,74.39,73.81,74.63,73.80,23.42K,0.30%
5157,2024-12-31,74.64,74.25,74.89,73.84,190.98K,0.34%


In [188]:
PRICE_COL = "Price"

In [189]:
# Select price column

brent = brent[["Date", PRICE_COL]].copy()

brent = brent.rename(columns={PRICE_COL: "price"})

brent.head()

,Date,price
0,2005-01-04,41.04
1,2005-01-05,40.51
2,2005-01-06,42.85
3,2005-01-07,43.18
4,2005-01-10,42.92


In [190]:
# Check missing values

print("Missing values before cleaning:")
print(brent.isna().sum())

brent = brent.dropna(subset=["Date", "price"]).copy()

print("\nMissing values after cleaning:")
print(brent.isna().sum())

Missing values before cleaning:
Date     0
price    0
dtype: int64

Missing values after cleaning:
Date     0
price    0
dtype: int64


In [191]:
# Dataset summary

print("Start date:", brent["Date"].min())
print("End date:", brent["Date"].max())
print("Number of observations:", len(brent))
print("Duplicated dates:", brent["Date"].duplicated().sum())

Start date: 2005-01-04 00:00:00
End date: 2024-12-31 00:00:00
Number of observations: 5158
Duplicated dates: 0


In [192]:
# Save cleaned raw price series

cleaned_price_path = os.path.join(DATA_PROCESSED_DIR, "d:\\Documents\\ESLSCA - MBA2 Finance de marché\\Machine Learning\\Projet LSTM\\repository\\brent-volatility-lstm-garch\\data\\processed\\brent_price_cleaned.csv")

brent.to_csv(cleaned_price_path, index=False)

print("Cleaned price data saved to:", cleaned_price_path)

Cleaned price data saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\data\processed\brent_price_cleaned.csv


## 3. Feature engineering

In this section, we construct the variables used for volatility forecasting.

The main transformations are:

- daily log returns;
- absolute returns;
- squared returns;
- past realized volatility over 10 trading days;
- past realized volatility over 20 trading days;
- future realized volatility over 10 trading days.

The target variable is the annualized 10-day future realized volatility.  
At date \(t\), explanatory variables only use information available up to \(t\), while the target uses returns from \(t+1\) to \(t+10\).

In [193]:
# Load cleaned price data

cleaned_price_path = os.path.join(DATA_PROCESSED_DIR, "brent_price_cleaned.csv")

brent = pd.read_csv(cleaned_price_path)

brent["Date"] = pd.to_datetime(brent["Date"])
brent = brent.sort_values("Date").reset_index(drop=True)

print("Dataset shape:", brent.shape)
print("Start date:", brent["Date"].min())
print("End date:", brent["Date"].max())

brent.head()

Dataset shape: (5158, 2)
Start date: 2005-01-04 00:00:00
End date: 2024-12-31 00:00:00


,Date,price
0,2005-01-04,41.04
1,2005-01-05,40.51
2,2005-01-06,42.85
3,2005-01-07,43.18
4,2005-01-10,42.92


In [194]:
# Daily log returns

brent["log_return"] = np.log(brent["price"] / brent["price"].shift(1))

brent[["Date", "price", "log_return"]].head()

,Date,price,log_return
0,2005-01-04,41.04,NaN
1,2005-01-05,40.51,-0.012998
2,2005-01-06,42.85,0.056157
3,2005-01-07,43.18,0.007672
4,2005-01-10,42.92,-0.006040


In [195]:
# Absolute and squared returns

brent["abs_return"] = brent["log_return"].abs()
brent["squared_return"] = brent["log_return"] ** 2

brent[["Date", "log_return", "abs_return", "squared_return"]].head()

,Date,log_return,abs_return,squared_return
0,2005-01-04,NaN,NaN,NaN
1,2005-01-05,-0.012998,0.012998,0.000169
2,2005-01-06,0.056157,0.056157,0.003154
3,2005-01-07,0.007672,0.007672,0.000059
4,2005-01-10,-0.006040,0.006040,0.000036


In [196]:
# Past realized volatility over 10 days

brent["rv_past_10d"] = np.sqrt(
    ANNUALIZATION_FACTOR
    * brent["squared_return"].rolling(window=10).mean()
)

brent[["Date", "rv_past_10d"]].head(15)

,Date,rv_past_10d
0,2005-01-04,NaN
1,2005-01-05,NaN
2,2005-01-06,NaN
3,2005-01-07,NaN
4,2005-01-10,NaN
5,2005-01-11,NaN
6,2005-01-12,NaN
7,2005-01-13,NaN
8,2005-01-14,NaN
9,2005-01-17,NaN


In [197]:
# Past realized volatility over 20 days

brent["rv_past_20d"] = np.sqrt(
    ANNUALIZATION_FACTOR
    * brent["squared_return"].rolling(window=20).mean()
)

brent[["Date", "rv_past_10d", "rv_past_20d"]].head(25)

,Date,rv_past_10d,rv_past_20d
0,2005-01-04,NaN,NaN
1,2005-01-05,NaN,NaN
2,2005-01-06,NaN,NaN
3,2005-01-07,NaN,NaN
4,2005-01-10,NaN,NaN
5,2005-01-11,NaN,NaN
6,2005-01-12,NaN,NaN
7,2005-01-13,NaN,NaN
8,2005-01-14,NaN,NaN
9,2005-01-17,NaN,NaN


In [198]:
# Future realized volatility over 10 days
# Target: uses returns from t+1 to t+10

h = FORECAST_HORIZON

brent["rv_future_10d"] = np.sqrt(
    ANNUALIZATION_FACTOR
    * brent["squared_return"].shift(-1).rolling(window=h).mean().shift(-(h - 1))
)

brent[["Date", "rv_future_10d"]].tail(15)

,Date,rv_future_10d
5143,2024-12-10,0.154686
5144,2024-12-11,0.126523
5145,2024-12-12,0.140684
5146,2024-12-13,0.120987
5147,2024-12-16,0.115680
5148,2024-12-17,NaN
5149,2024-12-18,NaN
5150,2024-12-19,NaN
5151,2024-12-20,NaN
5152,2024-12-23,NaN


In [199]:
# Temporal alignment check for one observation

check_idx = 50

date_t = brent.loc[check_idx, "Date"]
target_value = brent.loc[check_idx, "rv_future_10d"]

manual_target = np.sqrt(
    ANNUALIZATION_FACTOR
    * brent.loc[check_idx + 1: check_idx + h, "squared_return"].mean()
)

print("Date t:", date_t)
print("Computed target:", target_value)
print("Manual target:", manual_target)
print("Difference:", abs(target_value - manual_target))

Date t: 2005-03-15 00:00:00
Computed target: 0.3257922347529789
Manual target: 0.3257922347529789
Difference: 0.0


In [200]:
# Drop rows with missing values created by lags and rolling windows

feature_cols = [
    "log_return",
    "abs_return",
    "squared_return",
    "rv_past_10d",
    "rv_past_20d"
]

target_col = "rv_future_10d"

model_data = brent[
    ["Date", "price"] + feature_cols + [target_col]
].dropna().copy()

model_data = model_data.reset_index(drop=True)

print("Final dataset shape:", model_data.shape)
print("Start date:", model_data["Date"].min())
print("End date:", model_data["Date"].max())

model_data.head()

Final dataset shape: (5128, 8)
Start date: 2005-02-01 00:00:00
End date: 2024-12-16 00:00:00


,Date,price,log_return,abs_return,squared_return,rv_past_10d,rv_past_20d,rv_future_10d
0,2005-02-01,44.82,-0.024246,0.024246,5.878823e-04,0.315201,0.333101,0.224704
1,2005-02-02,44.01,-0.018238,0.018238,3.326096e-04,0.319361,0.336182,0.221491
2,2005-02-03,43.85,-0.003642,0.003642,1.326534e-05,0.316846,0.271017,0.223656
3,2005-02-04,43.89,0.000912,0.000912,8.313517e-07,0.275128,0.269665,0.230673
4,2005-02-07,43.04,-0.019557,0.019557,3.824601e-04,0.290507,0.277630,0.212937


In [201]:
# Missing values check

model_data.isna().sum()

Date              0
price             0
log_return        0
abs_return        0
squared_return    0
rv_past_10d       0
rv_past_20d       0
rv_future_10d     0
dtype: int64

In [202]:
# Save transformed dataset

processed_data_path = os.path.join(DATA_PROCESSED_DIR, "brent_model_data.csv")

model_data.to_csv(processed_data_path, index=False)

print("Processed model data saved to:", processed_data_path)

Processed model data saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\data\processed\brent_model_data.csv


## 4. Descriptive visualizations

In this section, we visualize the main time series used in the project:

- Brent futures closing price;
- daily log returns;
- realized volatility.

Interactive charts are created with Plotly for exploration, and static PNG figures are saved for inclusion in the final report.

In [203]:
# Brent price visualization

fig_price = go.Figure()

fig_price.add_trace(
    go.Scatter(
        x=model_data["Date"],
        y=model_data["price"],
        mode="lines",
        name="Brent price"
    )
)

fig_price.update_layout(
    title="Brent Crude Oil Futures Price",
    xaxis_title="Date",
    yaxis_title="Price",
    template="plotly_white",
    width=1000,
    height=500
)

fig_price.show()

price_fig_path = os.path.join(FIGURES_DIR, "price_brent.png")
fig_price.write_image(price_fig_path)

print("Figure saved to:", price_fig_path)

Figure saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\figures\price_brent.png


In [204]:
# Daily log returns visualization

fig_returns = go.Figure()

fig_returns.add_trace(
    go.Scatter(
        x=model_data["Date"],
        y=model_data["log_return"],
        mode="lines",
        name="Daily log returns"
    )
)

fig_returns.update_layout(
    title="Daily Log Returns of Brent Crude Oil Futures",
    xaxis_title="Date",
    yaxis_title="Log return",
    template="plotly_white",
    width=1000,
    height=500
)

fig_returns.show()

returns_fig_path = os.path.join(FIGURES_DIR, "log_returns.png")
fig_returns.write_image(returns_fig_path)

print("Figure saved to:", returns_fig_path)

Resorting to unclean kill browser.


Figure saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\figures\log_returns.png


In [205]:
# Future realized volatility visualization

fig_vol = go.Figure()

fig_vol.add_trace(
    go.Scatter(
        x=model_data["Date"],
        y=model_data["rv_future_10d"],
        mode="lines",
        name="10-day future realized volatility"
    )
)

fig_vol.update_layout(
    title="Annualized 10-Day Future Realized Volatility of Brent",
    xaxis_title="Date",
    yaxis_title="Annualized volatility",
    template="plotly_white",
    width=1000,
    height=500
)

fig_vol.show()

vol_fig_path = os.path.join(FIGURES_DIR, "realized_volatility.png")
fig_vol.write_image(vol_fig_path)

print("Figure saved to:", vol_fig_path)

Figure saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\figures\realized_volatility.png


In [206]:
fig_vol_compare = go.Figure()

fig_vol_compare.add_trace(
    go.Scatter(
        x=model_data["Date"],
        y=model_data["rv_past_10d"],
        mode="lines",
        name="Past 10-day realized volatility"
    )
)

fig_vol_compare.add_trace(
    go.Scatter(
        x=model_data["Date"],
        y=model_data["rv_future_10d"],
        mode="lines",
        name="Future 10-day realized volatility"
    )
)

fig_vol_compare.update_layout(
    title="Past and Future 10-Day Realized Volatility",
    xaxis_title="Date",
    yaxis_title="Annualized volatility",
    template="plotly_white",
    width=1000,
    height=500
)

fig_vol_compare.show()

## 5. Temporal train / validation / test split

The dataset is split chronologically into three subsets:

- training set: first 70% of observations;
- validation set: following 15% of observations;
- test set: final 15% of observations.

No random split is used, in order to preserve the temporal structure of the financial time series and avoid look-ahead bias.

In [207]:
# Ensure chronological order

model_data = model_data.sort_values("Date").reset_index(drop=True)

print("Start date:", model_data["Date"].min())
print("End date:", model_data["Date"].max())
print("Number of observations:", len(model_data))

Start date: 2005-02-01 00:00:00
End date: 2024-12-16 00:00:00
Number of observations: 5128


In [208]:
# Chronological train / validation / test split

n_obs = len(model_data)

train_end = int(n_obs * TRAIN_SIZE)
val_end = int(n_obs * (TRAIN_SIZE + VAL_SIZE))

train_data = model_data.iloc[:train_end].copy()
val_data = model_data.iloc[train_end:val_end].copy()
test_data = model_data.iloc[val_end:].copy()

print("Total observations:", n_obs)
print("Train observations:", len(train_data))
print("Validation observations:", len(val_data))
print("Test observations:", len(test_data))

Total observations: 5128
Train observations: 3589
Validation observations: 769
Test observations: 770


In [209]:
# Display split periods

split_summary = pd.DataFrame({
    "Subset": ["Train", "Validation", "Test"],
    "Start date": [
        train_data["Date"].min(),
        val_data["Date"].min(),
        test_data["Date"].min()
    ],
    "End date": [
        train_data["Date"].max(),
        val_data["Date"].max(),
        test_data["Date"].max()
    ],
    "Number of observations": [
        len(train_data),
        len(val_data),
        len(test_data)
    ]
})

split_summary

,Subset,Start date,End date,Number of observations
0,Train,2005-02-01,2019-01-02,3589
1,Validation,2019-01-03,2021-12-23,769
2,Test,2021-12-24,2024-12-16,770


In [210]:
# Check chronological consistency

assert train_data["Date"].max() < val_data["Date"].min(), "Train and validation periods overlap."
assert val_data["Date"].max() < test_data["Date"].min(), "Validation and test periods overlap."

print("Chronological split verified: no overlap between train, validation and test sets.")

Chronological split verified: no overlap between train, validation and test sets.


In [211]:
# Save split summary

split_summary_path = os.path.join(RESULTS_DIR, "split_summary.csv")

split_summary.to_csv(split_summary_path, index=False)

print("Split summary saved to:", split_summary_path)

Split summary saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\split_summary.csv


**Table — Chronological data split**

| Subset | Start date | End date | Number of observations |
|---|---:|---:|---:|
| Train | 2005-02-01 | 2019-01-02 | 3,589 |
| Validation | 2019-01-03 | 2021-12-23 | 769 |
| Test | 2021-12-24 | 2024-12-16 | 770 |

## 6. Naive baseline

The naive baseline assumes that future volatility is equal to recently observed volatility.

At each date \(t\), the forecast of the 10-day future realized volatility is defined as the past 10-day realized volatility:

\[
\widehat{RV}^{naive}_{t,t+10} = RV_{t-10,t}
\]

This baseline is used as a minimum benchmark. More complex models such as GARCH(1,1) and LSTM should outperform this simple persistence rule to justify their additional complexity.

In [212]:
# Naive baseline forecast on the test set

baseline_results = test_data[["Date", "rv_future_10d", "rv_past_10d"]].copy()

baseline_results = baseline_results.rename(
    columns={
        "rv_future_10d": "rv_realized",
        "rv_past_10d": "pred_naive"
    }
)

baseline_results.head()

,Date,rv_realized,pred_naive
4358,2021-12-24,0.232728,0.299421
4359,2021-12-27,0.177860,0.335454
4360,2021-12-28,0.247778,0.332882
4361,2021-12-29,0.253502,0.333165
4362,2021-12-30,0.253716,0.324227


In [213]:
# Unit check

baseline_results[["rv_realized", "pred_naive"]].describe()

,rv_realized,pred_naive
count,770.000000,770.000000
mean,0.325888,0.327851
std,0.145323,0.143958
min,0.102972,0.102972
25%,0.228137,0.232690
50%,0.301594,0.303411
75%,0.385243,0.385243
max,1.048188,1.048188


In [214]:
# Evaluation metrics for the naive baseline

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline_results["rv_realized"],
        baseline_results["pred_naive"]
    )
)

baseline_mae = mean_absolute_error(
    baseline_results["rv_realized"],
    baseline_results["pred_naive"]
)

print("Naive baseline RMSE:", baseline_rmse)
print("Naive baseline MAE:", baseline_mae)

Naive baseline RMSE: 0.136332211412619
Naive baseline MAE: 0.0979201000584484


In [215]:
# First metrics table

metrics = pd.DataFrame({
    "Model": ["Naive baseline"],
    "RMSE": [baseline_rmse],
    "MAE": [baseline_mae]
})

metrics

,Model,RMSE,MAE
0,Naive baseline,0.136332,0.09792


In [216]:
# Save naive baseline predictions

baseline_predictions_path = os.path.join(
    RESULTS_DIR,
    "baseline_predictions.csv"
)

baseline_results.to_csv(baseline_predictions_path, index=False)

print("Naive baseline predictions saved to:", baseline_predictions_path)

Naive baseline predictions saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\baseline_predictions.csv


In [217]:
# Save provisional metrics table

metrics_path = os.path.join(RESULTS_DIR, "metrics.csv")

metrics.to_csv(metrics_path, index=False)

print("Metrics saved to:", metrics_path)

Metrics saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\metrics.csv


## 7. GARCH(1,1) model

The GARCH(1,1) model is used as a classical econometric benchmark for volatility forecasting.

The model is estimated only on the training set, using daily log returns. Since the target variable is the annualized 10-day future realized volatility, the GARCH forecasts must be converted into a comparable 10-day annualized volatility forecast.

For each test date \(t\), the GARCH forecast is constructed as:

\[
\widehat{RV}^{GARCH}_{t,t+10}
=
\sqrt{
252 \times \frac{1}{10}
\sum_{h=1}^{10}
\widehat{\sigma}^{2}_{t+h|t}
}
\]

where \(\widehat{\sigma}^{2}_{t+h|t}\) is the conditional variance forecast at horizon \(h\).

In [218]:
# Prepare returns for GARCH estimation

garch_train_returns = train_data["log_return"] * 100

print("Number of training observations:", len(garch_train_returns))
print(garch_train_returns.describe())

Number of training observations: 3589
count    3589.000000
mean        0.004982
std         2.098141
min       -10.945524
25%        -1.072191
50%         0.041606
75%         1.060574
max        12.706595
Name: log_return, dtype: float64


In [219]:
# Estimate GARCH(1,1) model on training returns only

garch_model = arch_model(
    garch_train_returns,
    mean="Constant",
    vol="GARCH",
    p=1,
    q=1,
    dist="normal"
)

garch_res = garch_model.fit(disp="off")

print(garch_res.summary())

                     Constant Mean - GARCH Model Results                      
Dep. Variable:             log_return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -7258.40
Distribution:                  Normal   AIC:                           14524.8
Method:            Maximum Likelihood   BIC:                           14549.5
                                        No. Observations:                 3589
Date:                Mon, May 25 2026   Df Residuals:                     3588
Time:                        19:10:38   Df Model:                            1
                                  Mean Model                                 
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
mu             0.0323  2.738e-02      1.178      0.239 

In [220]:
# Check convergence

print("Convergence flag:", garch_res.convergence_flag)

if garch_res.convergence_flag == 0:
    print("GARCH estimation converged successfully.")
else:
    print("Warning: GARCH estimation may not have converged properly.")

Convergence flag: 0
GARCH estimation converged successfully.


In [221]:
# Extract estimated parameters

garch_params = garch_res.params

omega = garch_params.get("omega")
alpha = garch_params.get("alpha[1]")
beta = garch_params.get("beta[1]")

print("Estimated GARCH parameters:")
print("omega:", omega)
print("alpha:", alpha)
print("beta:", beta)
print("alpha + beta:", alpha + beta)

Estimated GARCH parameters:
omega: 0.021215391038119136
alpha: 0.05385103449790202
beta: 0.9423332337076799
alpha + beta: 0.9961842682055819


In [222]:
# GARCH out-of-sample forecasts
# We use the full return series but specify that the model estimation starts forecasting after the training period.

garch_all_returns = model_data["log_return"] * 100

garch_model_full_index = arch_model(
    garch_all_returns,
    mean="Constant",
    vol="GARCH",
    p=1,
    q=1,
    dist="normal"
)

garch_res_full_index = garch_model_full_index.fit(
    last_obs=train_data.index[-1],
    disp="off"
)

garch_forecasts = garch_res_full_index.forecast(
    horizon=FORECAST_HORIZON,
    start=test_data.index[0],
    reindex=True
)

garch_variance_forecasts = garch_forecasts.variance

garch_variance_forecasts.tail()

,h.01,h.02,h.03,h.04,h.05,h.06,h.07,h.08,h.09,h.10
5123,3.104911,3.114351,3.123754,3.133122,3.142454,3.151750,3.161011,3.170237,3.179428,3.188584
5124,3.120365,3.129745,3.139090,3.148399,3.157673,3.166912,3.176115,3.185284,3.194418,3.203517
5125,2.963110,2.973088,2.983028,2.992931,3.002796,3.012623,3.022414,3.032166,3.041882,3.051561
5126,2.923264,2.933394,2.943485,2.953538,2.963553,2.973530,2.983468,2.993369,3.003233,3.013058
5127,2.811362,2.821917,2.832432,2.842907,2.853343,2.863738,2.874094,2.884411,2.894689,2.904927


In [223]:
# Convert GARCH variance forecasts into annualized 10-day volatility forecasts

# Select only available horizon columns
available_horizon_cols = list(garch_variance_forecasts.columns)

print("Available GARCH forecast columns:", available_horizon_cols)

# Keep the columns corresponding to forecast horizons
horizon_cols = available_horizon_cols[:FORECAST_HORIZON]

print("Selected horizon columns:", horizon_cols)

# Select test-period forecasts
garch_test_variances = garch_variance_forecasts.loc[test_data.index, horizon_cols].copy()

# Convert variance from percentage squared to decimal squared
garch_test_variances_decimal = garch_test_variances / (100 ** 2)

# Aggregate the available horizon variance forecasts and annualize
garch_pred_10d = np.sqrt(
    ANNUALIZATION_FACTOR * garch_test_variances_decimal.mean(axis=1)
)

garch_results = test_data[["Date", "rv_future_10d"]].copy()
garch_results = garch_results.rename(columns={"rv_future_10d": "rv_realized"})
garch_results["pred_garch"] = garch_pred_10d.values

garch_results.head()

Available GARCH forecast columns: ['h.01', 'h.02', 'h.03', 'h.04', 'h.05', 'h.06', 'h.07', 'h.08', 'h.09', 'h.10']
Selected horizon columns: ['h.01', 'h.02', 'h.03', 'h.04', 'h.05', 'h.06', 'h.07', 'h.08', 'h.09', 'h.10']


,Date,rv_realized,pred_garch
4358,2021-12-24,0.232728,0.402089
4359,2021-12-27,0.177860,0.407733
4360,2021-12-28,0.247778,0.396886
4361,2021-12-29,0.253502,0.386304
4362,2021-12-30,0.253716,0.375875


In [224]:
# Unit check

garch_results[["rv_realized", "pred_garch"]].describe()

,rv_realized,pred_garch
count,770.000000,770.000000
mean,0.325888,0.345476
std,0.145323,0.107182
min,0.102972,0.183092
25%,0.228137,0.276928
50%,0.301594,0.332857
75%,0.385243,0.379222
max,1.048188,0.741909


In [225]:
# GARCH evaluation metrics

garch_rmse = np.sqrt(
    mean_squared_error(
        garch_results["rv_realized"],
        garch_results["pred_garch"]
    )
)

garch_mae = mean_absolute_error(
    garch_results["rv_realized"],
    garch_results["pred_garch"]
)

print("GARCH RMSE:", garch_rmse)
print("GARCH MAE:", garch_mae)

GARCH RMSE: 0.12455988705982027
GARCH MAE: 0.09017485015181825


In [226]:
# Save GARCH predictions

garch_predictions_path = os.path.join(
    RESULTS_DIR,
    "garch_predictions.csv"
)

garch_results.to_csv(garch_predictions_path, index=False)

print("GARCH predictions saved to:", garch_predictions_path)

GARCH predictions saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\garch_predictions.csv


In [227]:
# Add GARCH results to metrics table

garch_metrics = pd.DataFrame({
    "Model": ["GARCH(1,1)"],
    "RMSE": [garch_rmse],
    "MAE": [garch_mae]
})

metrics = pd.concat([metrics, garch_metrics], ignore_index=True)

metrics

,Model,RMSE,MAE
0,Naive baseline,0.136332,0.097920
1,"GARCH(1,1)",0.124560,0.090175


In [228]:
# Save updated metrics table

metrics_path = os.path.join(RESULTS_DIR, "metrics.csv")

metrics.to_csv(metrics_path, index=False)

print("Updated metrics saved to:", metrics_path)

Updated metrics saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\metrics.csv


## 8. LSTM data preparation

In this section, we prepare the data for the LSTM model.

The LSTM uses sequences of past observations to predict the annualized 10-day future realized volatility.

The input features are:

- daily log return;
- absolute return;
- squared return;
- past 10-day realized volatility;
- past 20-day realized volatility.

The target variable is the 10-day future realized volatility.

To avoid information leakage, the scalers are fitted only on the training set and then applied to the validation and test sets.

In [229]:
# LSTM features and target

feature_cols = [
    "log_return",
    "abs_return",
    "squared_return",
    "rv_past_10d",
    "rv_past_20d"
]

target_col = "rv_future_10d"

print("LSTM input features:", feature_cols)
print("Target variable:", target_col)

LSTM input features: ['log_return', 'abs_return', 'squared_return', 'rv_past_10d', 'rv_past_20d']
Target variable: rv_future_10d


In [230]:
# Extract raw feature and target arrays

X_train_raw = train_data[feature_cols].values
X_val_raw = val_data[feature_cols].values
X_test_raw = test_data[feature_cols].values

y_train_raw = train_data[[target_col]].values
y_val_raw = val_data[[target_col]].values
y_test_raw = test_data[[target_col]].values

print("X_train_raw shape:", X_train_raw.shape)
print("X_val_raw shape:", X_val_raw.shape)
print("X_test_raw shape:", X_test_raw.shape)

print("y_train_raw shape:", y_train_raw.shape)
print("y_val_raw shape:", y_val_raw.shape)
print("y_test_raw shape:", y_test_raw.shape)

X_train_raw shape: (3589, 5)
X_val_raw shape: (769, 5)
X_test_raw shape: (770, 5)
y_train_raw shape: (3589, 1)
y_val_raw shape: (769, 1)
y_test_raw shape: (770, 1)


In [231]:
# Feature scaling
# The scaler is fitted only on the training set to avoid information leakage.

scaler_X = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train_raw)
X_val_scaled = scaler_X.transform(X_val_raw)
X_test_scaled = scaler_X.transform(X_test_raw)

print("Feature scaling completed.")

Feature scaling completed.


In [232]:
# Target scaling
# The target scaler is fitted only on the training set.

scaler_y = StandardScaler()

y_train_scaled = scaler_y.fit_transform(y_train_raw)
y_val_scaled = scaler_y.transform(y_val_raw)
y_test_scaled = scaler_y.transform(y_test_raw)

print("Target scaling completed.")

Target scaling completed.


In [233]:
# Function to create LSTM sequences

def create_lstm_sequences(X, y, dates, sequence_length):
    """
    Create LSTM sequences.

    Each sequence ends at date t and is associated with the target value at date t.
    The target value corresponds to the future 10-day realized volatility from t+1 to t+10.
    """
    X_seq = []
    y_seq = []
    date_seq = []

    for i in range(sequence_length - 1, len(X)):
        X_seq.append(X[i - sequence_length + 1:i + 1])
        y_seq.append(y[i])
        date_seq.append(dates.iloc[i])

    return np.array(X_seq), np.array(y_seq), pd.Series(date_seq)

In [234]:
# Create LSTM sequences

seq_length = LSTM_SEQUENCE_LENGTH

X_train_seq, y_train_seq, train_seq_dates = create_lstm_sequences(
    X_train_scaled,
    y_train_scaled,
    train_data["Date"],
    seq_length
)

X_val_seq, y_val_seq, val_seq_dates = create_lstm_sequences(
    X_val_scaled,
    y_val_scaled,
    val_data["Date"],
    seq_length
)

X_test_seq, y_test_seq, test_seq_dates = create_lstm_sequences(
    X_test_scaled,
    y_test_scaled,
    test_data["Date"],
    seq_length
)

print("LSTM sequences created.")

LSTM sequences created.


In [235]:
# Tensor shape check

print("X_train_seq shape:", X_train_seq.shape)
print("y_train_seq shape:", y_train_seq.shape)

print("X_val_seq shape:", X_val_seq.shape)
print("y_val_seq shape:", y_val_seq.shape)

print("X_test_seq shape:", X_test_seq.shape)
print("y_test_seq shape:", y_test_seq.shape)

X_train_seq shape: (3560, 30, 5)
y_train_seq shape: (3560, 1)
X_val_seq shape: (740, 30, 5)
y_val_seq shape: (740, 1)
X_test_seq shape: (741, 30, 5)
y_test_seq shape: (741, 1)


In [236]:
# Sequence date check

sequence_summary = pd.DataFrame({
    "Subset": ["Train sequences", "Validation sequences", "Test sequences"],
    "Start date": [
        train_seq_dates.min(),
        val_seq_dates.min(),
        test_seq_dates.min()
    ],
    "End date": [
        train_seq_dates.max(),
        val_seq_dates.max(),
        test_seq_dates.max()
    ],
    "Number of sequences": [
        len(train_seq_dates),
        len(val_seq_dates),
        len(test_seq_dates)
    ]
})

sequence_summary

,Subset,Start date,End date,Number of sequences
0,Train sequences,2005-03-14,2019-01-02,3560
1,Validation sequences,2019-02-13,2021-12-23,740
2,Test sequences,2022-02-03,2024-12-16,741


In [237]:
# Check one LSTM sequence

example_idx = 0

print("First test sequence prediction date:", test_seq_dates.iloc[example_idx])
print("Sequence length:", X_test_seq[example_idx].shape[0])
print("Number of features:", X_test_seq[example_idx].shape[1])

print("Scaled target value:", y_test_seq[example_idx])
print("Original target value:", scaler_y.inverse_transform(y_test_seq[example_idx].reshape(1, -1))[0, 0])

First test sequence prediction date: 2022-02-03 00:00:00
Sequence length: 30
Number of features: 5
Scaled target value: [0.24391657]
Original target value: 0.3330525078891995


## 9. LSTM model training

In this section, we train the LSTM model.

The LSTM is used as a supervised regression model. It receives sequences of past observations and predicts the annualized 10-day future realized volatility.

The architecture is intentionally simple:

- one LSTM layer;
- one dropout layer;
- one dense output layer.

Early stopping is used to reduce the risk of overfitting.

In [238]:
# LSTM architecture

n_features = X_train_seq.shape[2]

lstm_model = Sequential([
    LSTM(
        units=64,
        input_shape=(LSTM_SEQUENCE_LENGTH, n_features)
    ),
    Dropout(0.2),
    Dense(1)
])

lstm_model.compile(
    optimizer="adam",
    loss="mse"
)

lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,985 (70.25 KB)

 Trainable params: 17,985 (70.25 KB)

 Non-trainable params: 0 (0.00 B)

In [239]:
# Early stopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [240]:
# Training parameters

BATCH_SIZE = 32
MAX_EPOCHS = 100

print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)

Batch size: 32
Maximum epochs: 100


In [241]:
# Train LSTM model

history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.4662 - val_loss: 2.4814
Epoch 2/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.3747 - val_loss: 2.3758
Epoch 3/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.3331 - val_loss: 2.4195
Epoch 4/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.3104 - val_loss: 2.6127
Epoch 5/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.2833 - val_loss: 2.7423
Epoch 6/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.2661 - val_loss: 2.8224
Epoch 7/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.2491 - val_loss: 2.7432
Epoch 8/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.2351 - val_loss: 2.7786
Epoch 9/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.2258 - val_loss: 2.8516
Epoch 10/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.2137 - val_loss: 2.7488
Epoch 11/100
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.1990 - val_loss: 2.6895
Epoch 12/100
112/112 ━━━━━━━━━

In [242]:
# Plot training and validation loss

fig_loss = go.Figure()

fig_loss.add_trace(
    go.Scatter(
        y=history.history["loss"],
        mode="lines",
        name="Training loss"
    )
)

fig_loss.add_trace(
    go.Scatter(
        y=history.history["val_loss"],
        mode="lines",
        name="Validation loss"
    )
)

fig_loss.update_layout(
    title="LSTM Training and Validation Loss",
    xaxis_title="Epoch",
    yaxis_title="MSE loss",
    template="plotly_white",
    width=900,
    height=500
)

fig_loss.show()

loss_fig_path = os.path.join(FIGURES_DIR, "lstm_training_loss.png")
fig_loss.write_image(loss_fig_path)

print("Training loss figure saved to:", loss_fig_path)

Training loss figure saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\figures\lstm_training_loss.png


In [243]:
# Number of epochs actually run

n_epochs_trained = len(history.history["loss"])

print("Number of epochs trained:", n_epochs_trained)
print("Minimum validation loss:", min(history.history["val_loss"]))

Number of epochs trained: 12
Minimum validation loss: 2.3758108615875244


In [244]:
# LSTM predictions on the test set

y_pred_lstm_scaled = lstm_model.predict(X_test_seq)

# Convert predictions and true values back to original scale
y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm_scaled)
y_test_lstm = scaler_y.inverse_transform(y_test_seq)

print("Predictions generated.")
print("y_pred_lstm shape:", y_pred_lstm.shape)
print("y_test_lstm shape:", y_test_lstm.shape)

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step
Predictions generated.
y_pred_lstm shape: (741, 1)
y_test_lstm shape: (741, 1)


In [245]:
# Build LSTM results table

lstm_results = pd.DataFrame({
    "Date": test_seq_dates.values,
    "rv_realized": y_test_lstm.flatten(),
    "pred_lstm": y_pred_lstm.flatten()
})

lstm_results.head()

,Date,rv_realized,pred_lstm
0,2022-02-03,0.333053,0.246057
1,2022-02-04,0.313098,0.244959
2,2022-02-07,0.326673,0.245187
3,2022-02-08,0.318631,0.255886
4,2022-02-09,0.315797,0.247834


In [246]:
# Unit and plausibility check

lstm_results[["rv_realized", "pred_lstm"]].describe()

,rv_realized,pred_lstm
count,741.000000,741.000000
mean,0.329077,0.324513
std,0.147089,0.088910
min,0.102972,0.182860
25%,0.228088,0.253676
50%,0.307334,0.312328
75%,0.389139,0.374707
max,1.048188,0.631532


In [247]:
# LSTM evaluation metrics

lstm_rmse = np.sqrt(
    mean_squared_error(
        lstm_results["rv_realized"],
        lstm_results["pred_lstm"]
    )
)

lstm_mae = mean_absolute_error(
    lstm_results["rv_realized"],
    lstm_results["pred_lstm"]
)

print("LSTM RMSE:", lstm_rmse)
print("LSTM MAE:", lstm_mae)

LSTM RMSE: 0.13125534027432387
LSTM MAE: 0.09016740864017186


In [248]:
# Save LSTM predictions

lstm_predictions_path = os.path.join(
    RESULTS_DIR,
    "lstm_predictions.csv"
)

lstm_results.to_csv(lstm_predictions_path, index=False)

print("LSTM predictions saved to:", lstm_predictions_path)

LSTM predictions saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\lstm_predictions.csv


In [249]:
# Add LSTM results to metrics table

lstm_metrics = pd.DataFrame({
    "Model": ["LSTM"],
    "RMSE": [lstm_rmse],
    "MAE": [lstm_mae]
})

metrics = pd.concat([metrics, lstm_metrics], ignore_index=True)

metrics_path = os.path.join(RESULTS_DIR, "metrics.csv")
metrics.to_csv(metrics_path, index=False)

metrics

,Model,RMSE,MAE
0,Naive baseline,0.136332,0.097920
1,"GARCH(1,1)",0.124560,0.090175
2,LSTM,0.131255,0.090167


In [250]:
# Save LSTM training history

history_df = pd.DataFrame(history.history)

history_path = os.path.join(RESULTS_DIR, "lstm_training_history.csv")
history_df.to_csv(history_path, index=False)

print("LSTM training history saved to:", history_path)

LSTM training history saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\lstm_training_history.csv


## 10. Align model predictions

In this section, we merge the predictions from the naive baseline, the GARCH(1,1) model and the LSTM model.

Because the LSTM uses sequences of 30 days, its predictions start later than the raw test set. Therefore, all models must be evaluated only on the common dates for which all forecasts are available.

The final prediction table contains:

- realized 10-day future volatility;
- naive baseline forecast;
- GARCH(1,1) forecast;
- LSTM forecast.

In [251]:
# Prepare prediction tables

baseline_for_merge = baseline_results[
    ["Date", "rv_realized", "pred_naive"]
].copy()

garch_for_merge = garch_results[
    ["Date", "pred_garch"]
].copy()

lstm_for_merge = lstm_results[
    ["Date", "pred_lstm"]
].copy()

# Ensure Date is datetime in all tables
baseline_for_merge["Date"] = pd.to_datetime(baseline_for_merge["Date"])
garch_for_merge["Date"] = pd.to_datetime(garch_for_merge["Date"])
lstm_for_merge["Date"] = pd.to_datetime(lstm_for_merge["Date"])

print("Baseline predictions:", baseline_for_merge.shape)
print("GARCH predictions:", garch_for_merge.shape)
print("LSTM predictions:", lstm_for_merge.shape)

Baseline predictions: (770, 3)
GARCH predictions: (770, 2)
LSTM predictions: (741, 2)


In [252]:
# Merge predictions on common dates

predictions = (
    baseline_for_merge
    .merge(garch_for_merge, on="Date", how="inner")
    .merge(lstm_for_merge, on="Date", how="inner")
)

predictions = predictions.sort_values("Date").reset_index(drop=True)

predictions.head()

,Date,rv_realized,pred_naive,pred_garch,pred_lstm
0,2022-02-03,0.333053,0.245529,0.282558,0.246057
1,2022-02-04,0.313098,0.270815,0.288151,0.244959
2,2022-02-07,0.326673,0.256123,0.281910,0.245187
3,2022-02-08,0.318631,0.253354,0.285523,0.255886
4,2022-02-09,0.315797,0.236957,0.279928,0.247834


In [253]:
# Drop missing values if any

print("Missing values before cleaning:")
print(predictions.isna().sum())

predictions = predictions.dropna().copy()
predictions = predictions.reset_index(drop=True)

print("\nMissing values after cleaning:")
print(predictions.isna().sum())

Missing values before cleaning:
Date           0
rv_realized    0
pred_naive     0
pred_garch     0
pred_lstm      0
dtype: int64

Missing values after cleaning:
Date           0
rv_realized    0
pred_naive     0
pred_garch     0
pred_lstm      0
dtype: int64


In [254]:
# Final aligned prediction period

print("Final aligned prediction table")
print("Number of observations:", len(predictions))
print("Start date:", predictions["Date"].min())
print("End date:", predictions["Date"].max())

predictions.tail()

Final aligned prediction table
Number of observations: 741
Start date: 2022-02-03 00:00:00
End date: 2024-12-16 00:00:00


,Date,rv_realized,pred_naive,pred_garch,pred_lstm
736,2024-12-10,0.154686,0.201993,0.281609,0.291580
737,2024-12-11,0.126523,0.221806,0.282287,0.284509
738,2024-12-12,0.140684,0.219769,0.275301,0.278774
739,2024-12-13,0.120987,0.230496,0.273502,0.269129
740,2024-12-16,0.115680,0.220776,0.268386,0.273147


In [255]:
# Consistency checks

assert predictions["Date"].is_monotonic_increasing, "Dates are not sorted."
assert predictions["Date"].duplicated().sum() == 0, "Duplicated dates found."

required_cols = [
    "Date",
    "rv_realized",
    "pred_naive",
    "pred_garch",
    "pred_lstm"
]

assert all(col in predictions.columns for col in required_cols), "Some required columns are missing."

print("Prediction table successfully aligned.")
print("All models will be evaluated on the same dates.")

Prediction table successfully aligned.
All models will be evaluated on the same dates.


In [256]:
# Save final aligned predictions

predictions_path = os.path.join(RESULTS_DIR, "predictions.csv")

predictions.to_csv(predictions_path, index=False)

print("Final aligned predictions saved to:", predictions_path)

Final aligned predictions saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\predictions.csv


## 11. Final model evaluation

In this section, we compute the final evaluation metrics for all models using the aligned prediction table.

All models are evaluated on the same dates.

The main metrics are:

- RMSE;
- MAE;
- QLIKE, only if all forecasts are strictly positive.

The final metrics table is saved in `results/metrics.csv`.

In [257]:
# Evaluation metric functions

def compute_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def compute_mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


def compute_qlike(y_true_vol, y_pred_vol, eps=1e-12):
    """
    QLIKE loss for volatility forecasts.

    Inputs are volatilities, not variances.
    The function converts volatilities into variances.

    QLIKE = mean(realized_variance / predicted_variance + log(predicted_variance))

    A small epsilon is used only for numerical safety.
    """
    realized_var = np.maximum(np.asarray(y_true_vol) ** 2, eps)
    predicted_var = np.maximum(np.asarray(y_pred_vol) ** 2, eps)

    return np.mean(realized_var / predicted_var + np.log(predicted_var))

In [258]:
# Positivity check for QLIKE

forecast_cols = ["pred_naive", "pred_garch", "pred_lstm"]

positivity_check = pd.DataFrame({
    "Model": ["Naive baseline", "GARCH(1,1)", "LSTM"],
    "Minimum forecast": [
        predictions["pred_naive"].min(),
        predictions["pred_garch"].min(),
        predictions["pred_lstm"].min()
    ],
    "Strictly positive": [
        (predictions["pred_naive"] > 0).all(),
        (predictions["pred_garch"] > 0).all(),
        (predictions["pred_lstm"] > 0).all()
    ]
})

positivity_check

,Model,Minimum forecast,Strictly positive
0,Naive baseline,0.102972,True
1,"GARCH(1,1)",0.183092,True
2,LSTM,0.182860,True


In [259]:
# Final evaluation on aligned prediction dates

models = {
    "Naive baseline": "pred_naive",
    "GARCH(1,1)": "pred_garch",
    "LSTM": "pred_lstm"
}

metrics_rows = []

for model_name, pred_col in models.items():
    y_true = predictions["rv_realized"]
    y_pred = predictions[pred_col]

    rmse_value = compute_rmse(y_true, y_pred)
    mae_value = compute_mae(y_true, y_pred)

    if (y_pred > 0).all():
        qlike_value = compute_qlike(y_true, y_pred)
    else:
        qlike_value = np.nan

    metrics_rows.append({
        "Model": model_name,
        "RMSE": rmse_value,
        "MAE": mae_value,
        "QLIKE": qlike_value
    })

final_metrics = pd.DataFrame(metrics_rows)

final_metrics

,Model,RMSE,MAE,QLIKE
0,Naive baseline,0.138458,0.099846,-1.018300
1,"GARCH(1,1)",0.125186,0.089989,-1.130319
2,LSTM,0.131255,0.090167,-1.057130


In [260]:
# Save final metrics table

metrics_path = os.path.join(RESULTS_DIR, "metrics.csv")

final_metrics.to_csv(metrics_path, index=False)

print("Final metrics saved to:", metrics_path)

Final metrics saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\metrics.csv


In [261]:
# Identify best models

best_rmse_model = final_metrics.loc[final_metrics["RMSE"].idxmin(), "Model"]
best_mae_model = final_metrics.loc[final_metrics["MAE"].idxmin(), "Model"]

best_rmse_value = final_metrics["RMSE"].min()
best_mae_value = final_metrics["MAE"].min()

print("Best model according to RMSE:", best_rmse_model, "| RMSE:", best_rmse_value)
print("Best model according to MAE:", best_mae_model, "| MAE:", best_mae_value)

Best model according to RMSE: GARCH(1,1) | RMSE: 0.1251858411277598
Best model according to MAE: GARCH(1,1) | MAE: 0.08998860835561594


In [262]:
# Relative improvement compared to the naive baseline

baseline_rmse_final = final_metrics.loc[
    final_metrics["Model"] == "Naive baseline", "RMSE"
].iloc[0]

baseline_mae_final = final_metrics.loc[
    final_metrics["Model"] == "Naive baseline", "MAE"
].iloc[0]

final_metrics["RMSE_improvement_vs_naive_%"] = (
    (baseline_rmse_final - final_metrics["RMSE"]) / baseline_rmse_final * 100
)

final_metrics["MAE_improvement_vs_naive_%"] = (
    (baseline_mae_final - final_metrics["MAE"]) / baseline_mae_final * 100
)

final_metrics

,Model,RMSE,MAE,QLIKE,RMSE_improvement_vs_naive_%,MAE_improvement_vs_naive_%
0,Naive baseline,0.138458,0.099846,-1.018300,0.000000,0.000000
1,"GARCH(1,1)",0.125186,0.089989,-1.130319,9.585747,9.872909
2,LSTM,0.131255,0.090167,-1.057130,5.202110,9.693833


In [263]:
# Save enriched final metrics table

final_metrics.to_csv(metrics_path, index=False)

print("Updated final metrics saved to:", metrics_path)

final_metrics

Updated final metrics saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\metrics.csv


,Model,RMSE,MAE,QLIKE,RMSE_improvement_vs_naive_%,MAE_improvement_vs_naive_%
0,Naive baseline,0.138458,0.099846,-1.018300,0.000000,0.000000
1,"GARCH(1,1)",0.125186,0.089989,-1.130319,9.585747,9.872909
2,LSTM,0.131255,0.090167,-1.057130,5.202110,9.693833


**Table — Final model performance on the aligned test set**

| Model | RMSE | MAE | QLIKE | RMSE improvement vs naive (%) | MAE improvement vs naive (%) |
|---|---:|---:|---:|---:|---:|
| Naive baseline | 0.138458 | 0.099846 | -1.018300 | 0.00 | 0.00 |
| GARCH(1,1) | 0.125186 | 0.089989 | -1.130319 | 9.59 | 9.87 |
| LSTM | 0.131255 | 0.090167 | -1.057130 | 5.20 | 9.69 |

## 12. Final forecast visualizations

In this section, we visualize the final forecasts on the aligned test set.

The figures compare:

- realized 10-day future volatility;
- naive baseline forecast;
- GARCH(1,1) forecast;
- LSTM forecast.

We also analyze the absolute forecast errors for each model.

In [264]:
# Forecast comparison on the aligned test set

fig_forecast = go.Figure()

fig_forecast.add_trace(
    go.Scatter(
        x=predictions["Date"],
        y=predictions["rv_realized"],
        mode="lines",
        name="Realized volatility"
    )
)

fig_forecast.add_trace(
    go.Scatter(
        x=predictions["Date"],
        y=predictions["pred_naive"],
        mode="lines",
        name="Naive baseline"
    )
)

fig_forecast.add_trace(
    go.Scatter(
        x=predictions["Date"],
        y=predictions["pred_garch"],
        mode="lines",
        name="GARCH(1,1)"
    )
)

fig_forecast.add_trace(
    go.Scatter(
        x=predictions["Date"],
        y=predictions["pred_lstm"],
        mode="lines",
        name="LSTM"
    )
)

fig_forecast.update_layout(
    title="Forecast Comparison on the Aligned Test Set",
    xaxis_title="Date",
    yaxis_title="Annualized 10-Day Realized Volatility",
    template="plotly_white",
    width=1100,
    height=550,
    legend_title="Series"
)

fig_forecast.show()

forecast_fig_path = os.path.join(FIGURES_DIR, "forecast_comparison.png")
fig_forecast.write_image(forecast_fig_path)

print("Forecast comparison figure saved to:", forecast_fig_path)

Forecast comparison figure saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\figures\forecast_comparison.png


In [265]:
# Absolute forecast errors

errors = predictions.copy()

errors["abs_error_naive"] = abs(errors["rv_realized"] - errors["pred_naive"])
errors["abs_error_garch"] = abs(errors["rv_realized"] - errors["pred_garch"])
errors["abs_error_lstm"] = abs(errors["rv_realized"] - errors["pred_lstm"])

fig_errors = go.Figure()

fig_errors.add_trace(
    go.Scatter(
        x=errors["Date"],
        y=errors["abs_error_naive"],
        mode="lines",
        name="Naive baseline"
    )
)

fig_errors.add_trace(
    go.Scatter(
        x=errors["Date"],
        y=errors["abs_error_garch"],
        mode="lines",
        name="GARCH(1,1)"
    )
)

fig_errors.add_trace(
    go.Scatter(
        x=errors["Date"],
        y=errors["abs_error_lstm"],
        mode="lines",
        name="LSTM"
    )
)

fig_errors.update_layout(
    title="Absolute Forecast Errors on the Aligned Test Set",
    xaxis_title="Date",
    yaxis_title="Absolute error",
    template="plotly_white",
    width=1100,
    height=550,
    legend_title="Model"
)

fig_errors.show()

errors_fig_path = os.path.join(FIGURES_DIR, "absolute_errors.png")
fig_errors.write_image(errors_fig_path)

print("Absolute errors figure saved to:", errors_fig_path)

Absolute errors figure saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\figures\absolute_errors.png


In [266]:
# Optional zoom on a stress period

stress_start = "2022-02-01"
stress_end = "2022-12-31"

stress_period = predictions[
    (predictions["Date"] >= stress_start) &
    (predictions["Date"] <= stress_end)
].copy()

print("Stress period observations:", len(stress_period))
print("Stress period start:", stress_period["Date"].min())
print("Stress period end:", stress_period["Date"].max())

Stress period observations: 235
Stress period start: 2022-02-03 00:00:00
Stress period end: 2022-12-30 00:00:00


In [267]:
# Forecast comparison during stress period

fig_stress = go.Figure()

fig_stress.add_trace(
    go.Scatter(
        x=stress_period["Date"],
        y=stress_period["rv_realized"],
        mode="lines",
        name="Realized volatility"
    )
)

fig_stress.add_trace(
    go.Scatter(
        x=stress_period["Date"],
        y=stress_period["pred_naive"],
        mode="lines",
        name="Naive baseline"
    )
)

fig_stress.add_trace(
    go.Scatter(
        x=stress_period["Date"],
        y=stress_period["pred_garch"],
        mode="lines",
        name="GARCH(1,1)"
    )
)

fig_stress.add_trace(
    go.Scatter(
        x=stress_period["Date"],
        y=stress_period["pred_lstm"],
        mode="lines",
        name="LSTM"
    )
)

fig_stress.update_layout(
    title="Forecast Comparison During the 2022 Stress Period",
    xaxis_title="Date",
    yaxis_title="Annualized 10-Day Realized Volatility",
    template="plotly_white",
    width=1100,
    height=550,
    legend_title="Series"
)

fig_stress.show()

stress_fig_path = os.path.join(FIGURES_DIR, "stress_period_zoom.png")
fig_stress.write_image(stress_fig_path)

print("Stress period zoom figure saved to:", stress_fig_path)

Resorting to unclean kill browser.


Stress period zoom figure saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\figures\stress_period_zoom.png


In [268]:
# Save absolute errors

errors_path = os.path.join(RESULTS_DIR, "absolute_errors.csv")

errors.to_csv(errors_path, index=False)

print("Absolute errors saved to:", errors_path)

Absolute errors saved to: d:\Documents\ESLSCA - MBA2 Finance de marché\Machine Learning\Projet LSTM\repository\brent-volatility-lstm-garch\results\absolute_errors.csv
